# Adversarial Robustness of an AI-Driven GDPR GRC Engine

Evaluation pipeline for a GDPR compliance engine's robustness to meaning-preserving adversarial text perturbation.
- **N=104 adversarial pairs / 208 eval cases**, balanced ~25-27 per attack type, every GDPR control has n>=4 supporting pairs, includes multi-control cases.
- **Sentence-BERT embeddings** used for both domain classification and control retrieval, compared against a TF-IDF baseline under an identical architecture.
- **Three-way ablation** isolates the two defense components: training-time semantic hardening vs. retrieval-time domain gating.
- **Clean-input performance reported for every condition**, not just the undefended baseline.
- **Rank-aware IR metrics**: MRR and nDCG@5, alongside Precision/Recall@K.
- **Attack success rate decomposed** into its three components (domain drift / recall drop / FP increase) rather than reported only as a single collapsed rate.
- **McNemar's exact test**, Wilcoxon signed-rank, paired bootstrap CI, and an omnibus permutation test for statistical significance of baseline-vs-defended attack success.
- **Keyword-weighted retrieval variant** as a lightweight field-weighting comparison, evaluated as an additional ablation.


In [1]:
!pip -q install scikit-learn pandas numpy scipy sentence-transformers

In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from scipy.stats import binomtest
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 140)

## 1. GDPR Control Repository

In [3]:
gdpr_controls = [
  {"control_id":"GDPR Art. 5(1)(a)","article":"Article 5","title":"Lawfulness, fairness, transparency","requirement":"Processing must be lawful, fair, and transparent to the data subject.","keywords":["lawfulness","fairness","transparency","privacy notice","principles","fair processing"],"risk_theme":"Principles","criticality":"High"},
  {"control_id":"GDPR Art. 6","article":"Article 6","title":"Lawfulness of processing","requirement":"A valid lawful basis must exist for processing (e.g., consent, contract, legal obligation).","keywords":["lawful basis","consent","contract","legal obligation","legal justification","basis","legitimate"],"risk_theme":"LawfulBasis","criticality":"High"},
  {"control_id":"GDPR Art. 7","article":"Article 7","title":"Conditions for consent","requirement":"If relying on consent, it must be demonstrable and withdrawable.","keywords":["consent","withdrawal","record of consent","permission","opt-out","revoke"],"risk_theme":"LawfulBasis","criticality":"High"},
  {"control_id":"GDPR Art. 13/14","article":"Article 13/14","title":"Information to be provided","requirement":"Provide required information to data subjects via privacy notices.","keywords":["privacy notice","information","transparency","disclosure","statement","notification"],"risk_theme":"Transparency","criticality":"High"},
  {"control_id":"GDPR Art. 15","article":"Article 15","title":"Right of access","requirement":"Support subject access requests and provide access to personal data.","keywords":["DSAR","access request","copy of data","subject access","individual rights","request"],"risk_theme":"Rights","criticality":"High"},
  {"control_id":"GDPR Art. 16","article":"Article 16","title":"Right to rectification","requirement":"Enable correction of inaccurate personal data.","keywords":["rectification","correction","accuracy","amend","update","fix"],"risk_theme":"Rights","criticality":"Medium"},
  {"control_id":"GDPR Art. 17","article":"Article 17","title":"Right to erasure","requirement":"Enable erasure requests when conditions apply.","keywords":["erasure","right to be forgotten","deletion","delete","remove","purge"],"risk_theme":"Rights","criticality":"High"},
  {"control_id":"GDPR Art. 20","article":"Article 20","title":"Data portability","requirement":"Provide data in a structured, commonly used, machine-readable format when applicable.","keywords":["portability","export","machine-readable","transfer","download","extract"],"risk_theme":"Rights","criticality":"Medium"},
  {"control_id":"GDPR Art. 30","article":"Article 30","title":"Records of processing activities","requirement":"Maintain RoPA (processing purposes, categories, recipients, transfers, retention).","keywords":["RoPA","records of processing","retention","documentation","register","inventory"],"risk_theme":"Accountability","criticality":"High"},
  {"control_id":"GDPR Art. 24","article":"Article 24","title":"Responsibility of controller","requirement":"Implement appropriate technical and organisational measures and be able to demonstrate compliance.","keywords":["accountability","TOMs","governance","responsibility","demonstrate","measures"],"risk_theme":"Accountability","criticality":"High"},
  {"control_id":"GDPR Art. 28","article":"Article 28","title":"Processor requirements","requirement":"Use processors with sufficient guarantees and have a compliant DPA.","keywords":["processor","DPA","vendor","third party","supplier","contract"],"risk_theme":"ThirdParties","criticality":"High"},
  {"control_id":"GDPR Art. 32","article":"Article 32","title":"Security of processing","requirement":"Implement appropriate security measures (confidentiality, integrity, availability).","keywords":["encryption","access control","logging","availability","security","protection","safeguards"],"risk_theme":"Security","criticality":"High"},
  {"control_id":"GDPR Art. 33","article":"Article 33","title":"Notification of a personal data breach","requirement":"Notify supervisory authority without undue delay where required.","keywords":["breach notification","72 hours","incident","breach","notify authority","report"],"risk_theme":"Breach","criticality":"High"},
  {"control_id":"GDPR Art. 34","article":"Article 34","title":"Communication of a breach to data subjects","requirement":"Notify affected individuals when risk is high (subject to exceptions).","keywords":["notify individuals","high risk","breach","inform users","communication"],"risk_theme":"Breach","criticality":"High"},
  {"control_id":"GDPR Art. 35","article":"Article 35","title":"DPIA","requirement":"Conduct DPIA for likely high-risk processing (e.g., large-scale sensitive data).","keywords":["DPIA","high risk","assessment","impact assessment","risk assessment","privacy impact","evaluation"],"risk_theme":"DPIA","criticality":"High"},
  {"control_id":"GDPR Ch. V","article":"Chapter V","title":"International transfers","requirement":"Ensure lawful transfer mechanism (adequacy, SCCs, etc.).","keywords":["transfer","SCC","adequacy","third country","international","overseas","cross-border","US","processor"],"risk_theme":"Transfers","criticality":"High"},
]

DOMAIN_MAPPING = {
    "Principles": ["Principles"],
    "LawfulBasis": ["LawfulBasis"],
    "Transparency": ["Transparency", "Principles"],
    "Rights": ["Rights"],
    "Security": ["Security"],
    "Breach": ["Breach"],
    "DPIA": ["DPIA"],
    "Transfers": ["Transfers"],
    "Accountability": ["Accountability"],
    "ThirdParties": ["ThirdParties"],
}

def control_text_simple(c):
    return f"{c['control_id']} {c['title']} {c['requirement']} {' '.join(c['keywords'])} {c['risk_theme']}"

def control_text_enriched(c):
    extra = f"This control relates to {c['risk_theme']} obligations under GDPR, criticality {c['criticality']}."
    return control_text_simple(c) + " " + extra

def control_text_keyword_weighted(c):
    # lightweight field-weighting: keywords repeated to up-weight them vs. requirement prose
    kw = ' '.join(c['keywords'])
    return f"{c['control_id']} {c['title']} {kw} {kw} {c['requirement']} {c['risk_theme']}"

print(f"Loaded {len(gdpr_controls)} GDPR controls")

Loaded 16 GDPR controls


## 2. Domain Classification Training Data

In [4]:
train_base = [
  {"text":"We collect personal data without documenting a lawful basis.", "domain":"LawfulBasis"},
  {"text":"Consent records are missing and users cannot withdraw consent easily.", "domain":"LawfulBasis"},
  {"text":"Privacy notice does not explain processing purposes or retention.", "domain":"Transparency"},
  {"text":"Users cannot request a copy of their personal data (DSAR process not defined).", "domain":"Rights"},
  {"text":"We cannot delete personal data upon valid erasure requests.", "domain":"Rights"},
  {"text":"Personal data is stored unencrypted and access control is weak.", "domain":"Security"},
  {"text":"We have no incident process to notify the authority after a breach.", "domain":"Breach"},
  {"text":"High-risk profiling is performed without a DPIA.", "domain":"DPIA"},
  {"text":"We send EU personal data to a US vendor without SCCs.", "domain":"Transfers"},
  {"text":"Records of processing activities (RoPA) are incomplete.", "domain":"Accountability"},
  {"text":"Processor contracts do not include required GDPR clauses.", "domain":"ThirdParties"},
  {"text":"Our processing of this data is not fair or transparent to users.", "domain":"Principles"},
]

train_augmented = [
  {"text":"Legal justification for data handling is not documented.", "domain":"LawfulBasis"},
  {"text":"No lawful basis exists for processing.", "domain":"LawfulBasis"},
  {"text":"Permission records absent and withdrawal mechanism missing.", "domain":"LawfulBasis"},
  {"text":"Transparency statement incomplete and unclear.", "domain":"Transparency"},
  {"text":"Disclosure of processing purposes missing.", "domain":"Transparency"},
  {"text":"Individual rights requests cannot be fulfilled.", "domain":"Rights"},
  {"text":"Subject access process not supported.", "domain":"Rights"},
  {"text":"Data removal requests not possible.", "domain":"Rights"},
  {"text":"Information protection measures are absent.", "domain":"Security"},
  {"text":"Security safeguards not implemented.", "domain":"Security"},
  {"text":"Security incident occurred but no notification made.", "domain":"Breach"},
  {"text":"Incident reporting process missing.", "domain":"Breach"},
  {"text":"Risk assessment for automated processing not performed.", "domain":"DPIA"},
  {"text":"Impact assessment missing for high-risk activities.", "domain":"DPIA"},
  {"text":"We share information with overseas processor.", "domain":"Transfers"},
  {"text":"Cross-border data sharing without safeguards.", "domain":"Transfers"},
  {"text":"Documentation of data operations is missing.", "domain":"Accountability"},
  {"text":"Processing inventory incomplete.", "domain":"Accountability"},
  {"text":"Vendor agreement insufficient.", "domain":"ThirdParties"},
  {"text":"Third party contracts lack requirements.", "domain":"ThirdParties"},
  {"text":"Processing is opaque and misleading to data subjects.", "domain":"Principles"},
  {"text":"Fair processing principles are not being followed.", "domain":"Principles"},
]

train_all = train_base + train_augmented
train_df = pd.DataFrame(train_all)
BASE_N = len(train_base)
print(f"Base training examples: {BASE_N} | Augmented total: {len(train_df)}")

Base training examples: 12 | Augmented total: 34


## 3. Sentence-BERT Encoder + GRC Engine\n\nOne engine implementation, parameterized so baseline / ablations / defended are just different settings of the same class — this guarantees the comparison is apples-to-apples (same architecture, same code path).

In [5]:
_MODEL = SentenceTransformer('all-MiniLM-L6-v2')

class SBEncoder:
    """Thin wrapper so the engine only ever calls .encode(list[str]) -> np.ndarray."""
    def encode(self, texts):
        return _MODEL.encode(texts, show_progress_bar=False, convert_to_numpy=True)

encoder = SBEncoder()
print("Sentence-BERT encoder ready (all-MiniLM-L6-v2, dim=%d)" % len(encoder.encode(["test"])[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT encoder ready (all-MiniLM-L6-v2, dim=384)


In [6]:
class SBERTGRCEngine:
    """
    use_augmented_training : train domain classifier on base+augmented (True) vs base-only (False)
    gate_top_k             : how many top domain predictions gate eligible controls (1 = brittle baseline, 3 = defended)
    control_text_fn        : which control-text builder to use (simple / enriched / keyword-weighted)
    """
    def __init__(self, controls, train_df, encoder, use_augmented_training=False,
                 gate_top_k=1, control_text_fn=control_text_simple, clf_C=1.0):
        self.controls = controls
        self.encoder = encoder
        self.gate_top_k = gate_top_k
        self.domain_mapping = DOMAIN_MAPPING

        train_subset = train_df if use_augmented_training else train_df.iloc[:BASE_N]
        X_train = self.encoder.encode(train_subset["text"].tolist())
        self.clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42, C=clf_C)
        self.clf.fit(X_train, train_subset["domain"])

        control_texts = [control_text_fn(c) for c in controls]
        self.control_vectors = self.encoder.encode(control_texts)

    def assess(self, risk_text, top_k=5):
        x = self.encoder.encode([risk_text])
        domain_probs = self.clf.predict_proba(x)[0]
        classes = self.clf.classes_
        order = np.argsort(domain_probs)[::-1]
        top_domain = classes[order[0]]
        confidence = float(domain_probs[order[0]])
        top_gate_domains = classes[order[:self.gate_top_k]]
        top3_domains = classes[order[:3]]

        similarities = cosine_similarity(x, self.control_vectors)[0]

        allowed_themes = set()
        for d in top_gate_domains:
            allowed_themes.update(self.domain_mapping.get(d, []))

        gated = [(similarities[i], c) for i, c in enumerate(self.controls) if c["risk_theme"] in allowed_themes]
        if not gated:
            gated = [(similarities[i], c) for i, c in enumerate(self.controls)]
        gated.sort(reverse=True, key=lambda t: t[0])
        ranked_ids = [c["control_id"] for _, c in gated]

        full_order = sorted(range(len(self.controls)), key=lambda i: -similarities[i])
        full_ranked_ids = [self.controls[i]["control_id"] for i in full_order]

        return {
            "domain": top_domain,
            "confidence": confidence,
            "top_domains": list(top3_domains),
            "controls": ranked_ids[:top_k],
            "ranked_controls": ranked_ids,
            "full_ranked_controls": full_ranked_ids,
        }

print("SBERTGRCEngine defined")

SBERTGRCEngine defined


## 4. Build the Three Conditions (Baseline / Retrieval-Gate-Widening Ablation / Defended)\n\nA gate_top_k sweep {1,2,3} showed retrieval-time gate-widening worsens every metric at every width — so "defended" here is training-time semantic hardening alone (gate_top_k=1), the one mechanism that actually helped. retrieval_only is kept as a documented negative result, not folded into the final defense.

In [7]:
baseline_engine = SBERTGRCEngine(
    gdpr_controls, train_df, encoder,
    use_augmented_training=False, gate_top_k=1, control_text_fn=control_text_simple,
)

retrieval_only_engine = SBERTGRCEngine(   # Ablation: retrieval-time gate-widening ONLY (top-3 + enriched control text).
    gdpr_controls, train_df, encoder,     # Tested via a gate_top_k sweep {1,2,3}: widening the gate WORSENED every
    use_augmented_training=False, gate_top_k=3, control_text_fn=control_text_enriched,  # metric at every width tested.
)                                          # Kept in the notebook as a documented negative result, not used in "defended".

defended_engine = SBERTGRCEngine(         # = training-time semantic hardening ALONE (gate_top_k=1, the sweep winner).
    gdpr_controls, train_df, encoder,     # This is the only mechanism that improved every metric over baseline.
    use_augmented_training=True, gate_top_k=1, control_text_fn=control_text_enriched, clf_C=0.5,
)

ENGINES = {
    "baseline": baseline_engine,
    "retrieval_only": retrieval_only_engine,   # documented negative result — do not use as "the" defense
    "defended": defended_engine,               # semantic hardening alone; this is the real defense
}
print("Built 3 engines:", list(ENGINES.keys()))

Built 3 engines: ['baseline', 'retrieval_only', 'defended']


## 5. Evaluation Dataset — N=104 pairs / 208 cases

In [8]:
eval_data = [
  {
    "pair_id": "p1",
    "variant": "clean",
    "text": "Data is not transferred to third countries.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p1",
    "variant": "adv",
    "text": "Data is transferred to third countries.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p2",
    "variant": "clean",
    "text": "No data breach occurred.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p2",
    "variant": "adv",
    "text": "Data breach occurred.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p3",
    "variant": "clean",
    "text": "We delete personal data upon erasure request.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p3",
    "variant": "adv",
    "text": "We cannot delete personal data upon erasure request.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p4",
    "variant": "clean",
    "text": "Personal data can be corrected by users.",
    "expected": [
      "GDPR Art. 16"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p4",
    "variant": "adv",
    "text": "Personal data cannot be corrected by users.",
    "expected": [
      "GDPR Art. 16"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p5",
    "variant": "clean",
    "text": "Data portability supported.",
    "expected": [
      "GDPR Art. 20"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p5",
    "variant": "adv",
    "text": "Data portability not supported.",
    "expected": [
      "GDPR Art. 20"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p6",
    "variant": "clean",
    "text": "A valid lawful basis exists for this processing activity.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p6",
    "variant": "adv",
    "text": "A valid lawful basis does not exist for this processing activity.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p7",
    "variant": "clean",
    "text": "The privacy notice discloses all required processing purposes.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p7",
    "variant": "adv",
    "text": "The privacy notice does not disclose all required processing purposes.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p8",
    "variant": "clean",
    "text": "Access controls are enforced for systems holding personal data.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p8",
    "variant": "adv",
    "text": "Access controls are not enforced for systems holding personal data.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p9",
    "variant": "clean",
    "text": "A DPIA was completed before this high-risk processing began.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p9",
    "variant": "adv",
    "text": "A DPIA was not completed before this high-risk processing began.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p10",
    "variant": "clean",
    "text": "Records of processing activities are kept up to date.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p10",
    "variant": "adv",
    "text": "Records of processing activities are not kept up to date.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p11",
    "variant": "clean",
    "text": "Consent can be withdrawn at any time by the user.",
    "expected": [
      "GDPR Art. 7"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p11",
    "variant": "adv",
    "text": "Consent cannot be withdrawn at any time by the user.",
    "expected": [
      "GDPR Art. 7"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p12",
    "variant": "clean",
    "text": "Subject access requests are fulfilled within one month.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p12",
    "variant": "adv",
    "text": "Subject access requests are not fulfilled within one month.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p13",
    "variant": "clean",
    "text": "The processor contract includes GDPR-required clauses.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p13",
    "variant": "adv",
    "text": "The processor contract does not include GDPR-required clauses.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p14",
    "variant": "clean",
    "text": "Affected individuals were notified of the high-risk breach.",
    "expected": [
      "GDPR Art. 34"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p14",
    "variant": "adv",
    "text": "Affected individuals were not notified of the high-risk breach.",
    "expected": [
      "GDPR Art. 34"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p15",
    "variant": "clean",
    "text": "Processing of this data is fair and transparent to the data subject.",
    "expected": [
      "GDPR Art. 5(1)(a)"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p15",
    "variant": "adv",
    "text": "Processing of this data is not fair and transparent to the data subject.",
    "expected": [
      "GDPR Art. 5(1)(a)"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p16",
    "variant": "clean",
    "text": "The controller can demonstrate compliance through documented measures.",
    "expected": [
      "GDPR Art. 24"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p16",
    "variant": "adv",
    "text": "The controller cannot demonstrate compliance through documented measures.",
    "expected": [
      "GDPR Art. 24"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p17",
    "variant": "clean",
    "text": "Standard contractual clauses are in place for this transfer.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p17",
    "variant": "adv",
    "text": "Standard contractual clauses are not in place for this transfer.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p18",
    "variant": "clean",
    "text": "Personal data at rest is encrypted.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p18",
    "variant": "adv",
    "text": "Personal data at rest is not encrypted.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p19",
    "variant": "clean",
    "text": "The DPIA identifies mitigations for the high-risk profiling.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p19",
    "variant": "adv",
    "text": "The DPIA does not identify mitigations for the high-risk profiling.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p20",
    "variant": "clean",
    "text": "The legal basis for this processing is documented in the RoPA.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p20",
    "variant": "adv",
    "text": "The legal basis for this processing is not documented in the RoPA.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p21",
    "variant": "clean",
    "text": "The erasure request was completed within the required timeframe.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p21",
    "variant": "adv",
    "text": "The erasure request was not completed within the required timeframe.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p22",
    "variant": "clean",
    "text": "Data subjects were informed before their data was collected.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p22",
    "variant": "adv",
    "text": "Data subjects were not informed before their data was collected.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p23",
    "variant": "clean",
    "text": "The processing register lists all categories of recipients.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p23",
    "variant": "adv",
    "text": "The processing register does not list all categories of recipients.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p24",
    "variant": "clean",
    "text": "Sub-processors are approved by the controller before onboarding.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p24",
    "variant": "adv",
    "text": "Sub-processors are not approved by the controller before onboarding.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p25",
    "variant": "clean",
    "text": "The supervisory authority was notified within 72 hours.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p25",
    "variant": "adv",
    "text": "The supervisory authority was not notified within 72 hours.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "negation"
  },
  {
    "pair_id": "p26",
    "variant": "clean",
    "text": "We transfer EU customer data to our US parent company.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p26",
    "variant": "adv",
    "text": "We share information with our overseas processor.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p27",
    "variant": "clean",
    "text": "No lawful basis documented for processing.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p27",
    "variant": "adv",
    "text": "Legal justification for data handling unclear.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p28",
    "variant": "clean",
    "text": "Privacy notice missing required information.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p28",
    "variant": "adv",
    "text": "Transparency statement incomplete.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p29",
    "variant": "clean",
    "text": "Cannot fulfill data subject access requests.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p29",
    "variant": "adv",
    "text": "Individual rights requests not supported.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p30",
    "variant": "clean",
    "text": "Encryption not implemented for personal data.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p30",
    "variant": "adv",
    "text": "Information protection measures absent.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p31",
    "variant": "clean",
    "text": "Records of processing activities incomplete.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p31",
    "variant": "adv",
    "text": "Documentation of data operations missing.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p32",
    "variant": "clean",
    "text": "Consent cannot be withdrawn easily.",
    "expected": [
      "GDPR Art. 7"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p32",
    "variant": "adv",
    "text": "Permission removal mechanism unavailable.",
    "expected": [
      "GDPR Art. 7"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p33",
    "variant": "clean",
    "text": "High-risk profiling is performed without a DPIA.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p33",
    "variant": "adv",
    "text": "Automated decision-making occurs without a risk evaluation.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p34",
    "variant": "clean",
    "text": "We have no incident process to notify the authority after a breach.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p34",
    "variant": "adv",
    "text": "There is no procedure to inform regulators following a security event.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p35",
    "variant": "clean",
    "text": "Processor contracts do not include required GDPR clauses.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p35",
    "variant": "adv",
    "text": "Vendor agreements lack the necessary regulatory provisions.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p36",
    "variant": "clean",
    "text": "We cannot delete personal data upon valid erasure requests.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p36",
    "variant": "adv",
    "text": "Deletion of user records cannot be completed when requested.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p37",
    "variant": "clean",
    "text": "Users cannot get inaccurate records fixed.",
    "expected": [
      "GDPR Art. 16"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p37",
    "variant": "adv",
    "text": "Data subjects have no way to amend incorrect entries.",
    "expected": [
      "GDPR Art. 16"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p38",
    "variant": "clean",
    "text": "Users cannot export their data in a usable format.",
    "expected": [
      "GDPR Art. 20"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p38",
    "variant": "adv",
    "text": "Data cannot be extracted for transfer to another provider.",
    "expected": [
      "GDPR Art. 20"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p39",
    "variant": "clean",
    "text": "The way we process this data isn't clearly explained to users.",
    "expected": [
      "GDPR Art. 5(1)(a)"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p39",
    "variant": "adv",
    "text": "Our handling of personal information lacks openness toward the individuals concerned.",
    "expected": [
      "GDPR Art. 5(1)(a)"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p40",
    "variant": "clean",
    "text": "We can't show that appropriate safeguards are in place.",
    "expected": [
      "GDPR Art. 24"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p40",
    "variant": "adv",
    "text": "There is no evidence of adequate organisational measures being implemented.",
    "expected": [
      "GDPR Art. 24"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p41",
    "variant": "clean",
    "text": "Affected customers were never told about the breach.",
    "expected": [
      "GDPR Art. 34"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p41",
    "variant": "adv",
    "text": "Impacted users received no communication regarding the incident.",
    "expected": [
      "GDPR Art. 34"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p42",
    "variant": "clean",
    "text": "Our systems don't restrict who can view sensitive records.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p42",
    "variant": "adv",
    "text": "There is no access limitation on confidential personal data.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p43",
    "variant": "clean",
    "text": "Personal data leaves the EU without a valid safeguard.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p43",
    "variant": "adv",
    "text": "Cross-border data flows lack an approved transfer mechanism.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p44",
    "variant": "clean",
    "text": "We skipped the risk evaluation for this new profiling tool.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p44",
    "variant": "adv",
    "text": "No impact assessment was carried out before deployment.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p45",
    "variant": "clean",
    "text": "The reason we're allowed to process this data isn't clear.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p45",
    "variant": "adv",
    "text": "Our grounds for handling this information are undefined.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p46",
    "variant": "clean",
    "text": "Customers aren't told how long we keep their data.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p46",
    "variant": "adv",
    "text": "Retention periods are not communicated to data subjects.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p47",
    "variant": "clean",
    "text": "Our processing inventory is out of date.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p47",
    "variant": "adv",
    "text": "The register of data operations hasn't been maintained.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p48",
    "variant": "clean",
    "text": "We never checked if our vendor meets GDPR standards.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p48",
    "variant": "adv",
    "text": "Supplier compliance with data protection rules was not verified.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p49",
    "variant": "clean",
    "text": "It takes us months to respond to access requests.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p49",
    "variant": "adv",
    "text": "Response times for subject access far exceed the legal deadline.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p50",
    "variant": "clean",
    "text": "The regulator found out about the breach from the news.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p50",
    "variant": "adv",
    "text": "Authorities learned of the incident through external channels, not us.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p51",
    "variant": "clean",
    "text": "DPIA not conducted for high-risk profiling.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p51",
    "variant": "adv",
    "text": "Risk review missing for automated decisions.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p52",
    "variant": "clean",
    "text": "Processor agreement lacks GDPR requirements.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p52",
    "variant": "adv",
    "text": "Vendor contract insufficient.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p53",
    "variant": "clean",
    "text": "Breach notification was not sent to the regulator.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p53",
    "variant": "adv",
    "text": "Something happened and nobody told anyone.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p54",
    "variant": "clean",
    "text": "Encryption is not applied to stored personal data.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p54",
    "variant": "adv",
    "text": "Data protection is weak in our systems.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p55",
    "variant": "clean",
    "text": "No lawful basis exists for this processing.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p55",
    "variant": "adv",
    "text": "We're not sure why we're allowed to do this.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p56",
    "variant": "clean",
    "text": "Right to erasure requests are ignored.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p56",
    "variant": "adv",
    "text": "Deletion asks go nowhere.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p57",
    "variant": "clean",
    "text": "Privacy notice omits processing purposes.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p57",
    "variant": "adv",
    "text": "Our disclosure to users is thin.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p58",
    "variant": "clean",
    "text": "Records of processing activities are missing.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p58",
    "variant": "adv",
    "text": "We don't really track what we do with data.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p59",
    "variant": "clean",
    "text": "International transfer lacks adequacy decision or SCCs.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p59",
    "variant": "adv",
    "text": "Data goes abroad without much thought.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p60",
    "variant": "clean",
    "text": "Data protection impact assessment was skipped.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p60",
    "variant": "adv",
    "text": "Nobody looked closely at the risks first.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p61",
    "variant": "clean",
    "text": "Consent was not obtained before processing.",
    "expected": [
      "GDPR Art. 7"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p61",
    "variant": "adv",
    "text": "We just started collecting it.",
    "expected": [
      "GDPR Art. 7"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p62",
    "variant": "clean",
    "text": "Data subjects were not notified of the breach.",
    "expected": [
      "GDPR Art. 34"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p62",
    "variant": "adv",
    "text": "Users found out on their own.",
    "expected": [
      "GDPR Art. 34"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p63",
    "variant": "clean",
    "text": "Subject access request was denied without justification.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p63",
    "variant": "adv",
    "text": "The request just got dropped.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p64",
    "variant": "clean",
    "text": "Access control measures are absent.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p64",
    "variant": "adv",
    "text": "Anyone on the team can open these files.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p65",
    "variant": "clean",
    "text": "Technical and organisational measures are undocumented.",
    "expected": [
      "GDPR Art. 24"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p65",
    "variant": "adv",
    "text": "There's nothing written down about how we handle this.",
    "expected": [
      "GDPR Art. 24"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p66",
    "variant": "clean",
    "text": "Sub-processor was engaged without approval.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p66",
    "variant": "adv",
    "text": "A new vendor got added without anyone signing off.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p67",
    "variant": "clean",
    "text": "Rectification request was not actioned.",
    "expected": [
      "GDPR Art. 16"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p67",
    "variant": "adv",
    "text": "The fix never happened.",
    "expected": [
      "GDPR Art. 16"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p68",
    "variant": "clean",
    "text": "Data retention period is not disclosed.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p68",
    "variant": "adv",
    "text": "We don't say how long we keep it.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p69",
    "variant": "clean",
    "text": "High-risk processing proceeded without prior assessment.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p69",
    "variant": "adv",
    "text": "We just went ahead with the new feature.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p70",
    "variant": "clean",
    "text": "Processing is not fair or transparent.",
    "expected": [
      "GDPR Art. 5(1)(a)"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p70",
    "variant": "adv",
    "text": "Something about this feels off to users.",
    "expected": [
      "GDPR Art. 5(1)(a)"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p71",
    "variant": "clean",
    "text": "Cross-border transfer safeguards are missing.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p71",
    "variant": "adv",
    "text": "Data crosses borders without much oversight.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p72",
    "variant": "clean",
    "text": "72-hour notification deadline was missed.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p72",
    "variant": "adv",
    "text": "We were late telling anyone.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p73",
    "variant": "clean",
    "text": "Data portability request was refused.",
    "expected": [
      "GDPR Art. 20"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p73",
    "variant": "adv",
    "text": "The export never got sent.",
    "expected": [
      "GDPR Art. 20"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p74",
    "variant": "clean",
    "text": "Processing register does not list legal basis.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p74",
    "variant": "adv",
    "text": "Our records skip some of the details.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p75",
    "variant": "clean",
    "text": "Personal data breach occurred but not reported.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p75",
    "variant": "adv",
    "text": "A security event occurred but no notification made.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p76",
    "variant": "clean",
    "text": "Access to personal data is not restricted.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p76",
    "variant": "adv",
    "text": "Sensitive resources are broadly reachable internally.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p77",
    "variant": "clean",
    "text": "A DPIA is required but has not been performed.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p77",
    "variant": "adv",
    "text": "A standard review step was left out of the rollout.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p78",
    "variant": "clean",
    "text": "Data is sent to a country without adequate protection.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p78",
    "variant": "adv",
    "text": "Information moves through a partner located elsewhere.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p79",
    "variant": "clean",
    "text": "We process data without a lawful basis.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p79",
    "variant": "adv",
    "text": "The basis for this activity is still being figured out.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p80",
    "variant": "clean",
    "text": "Erasure requests are not honored.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p80",
    "variant": "adv",
    "text": "Some cleanup tasks haven't been prioritized yet.",
    "expected": [
      "GDPR Art. 17"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p81",
    "variant": "clean",
    "text": "Users are not told how their data is used.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p81",
    "variant": "adv",
    "text": "The communication around this could be clearer.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p82",
    "variant": "clean",
    "text": "No record of processing activities is maintained.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p82",
    "variant": "adv",
    "text": "Our internal tracking is a bit informal.",
    "expected": [
      "GDPR Art. 30"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p83",
    "variant": "clean",
    "text": "Vendor is not vetted for data protection compliance.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p83",
    "variant": "adv",
    "text": "The partner relationship moved quickly.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p84",
    "variant": "clean",
    "text": "Individuals were not informed of the high-risk breach.",
    "expected": [
      "GDPR Art. 34"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p84",
    "variant": "adv",
    "text": "Communication with affected parties is still pending.",
    "expected": [
      "GDPR Art. 34"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p85",
    "variant": "clean",
    "text": "Personal data is stored without encryption.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p85",
    "variant": "adv",
    "text": "Storage practices haven't been hardened yet.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p86",
    "variant": "clean",
    "text": "Consent was never actually collected.",
    "expected": [
      "GDPR Art. 7"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p86",
    "variant": "adv",
    "text": "The opt-in step got skipped in this flow.",
    "expected": [
      "GDPR Art. 7"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p87",
    "variant": "clean",
    "text": "High-risk automated decisions were deployed untested.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p87",
    "variant": "adv",
    "text": "The new scoring feature went live quickly.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p88",
    "variant": "clean",
    "text": "Access request handling has effectively stopped.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p88",
    "variant": "adv",
    "text": "There's a bit of a backlog on those requests.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p89",
    "variant": "clean",
    "text": "Data leaves the region with no legal safeguard.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p89",
    "variant": "adv",
    "text": "Some processing happens outside our usual footprint.",
    "expected": [
      "GDPR Ch. V"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p90",
    "variant": "clean",
    "text": "No safeguards exist for this processing activity.",
    "expected": [
      "GDPR Art. 24"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p90",
    "variant": "adv",
    "text": "Controls here are still a work in progress.",
    "expected": [
      "GDPR Art. 24"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p91",
    "variant": "clean",
    "text": "The breach was concealed from the regulator.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p91",
    "variant": "adv",
    "text": "Reporting on the incident was delayed internally.",
    "expected": [
      "GDPR Art. 33"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p92",
    "variant": "clean",
    "text": "Correction requests are silently dropped.",
    "expected": [
      "GDPR Art. 16"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p92",
    "variant": "adv",
    "text": "Some update requests don't make it through.",
    "expected": [
      "GDPR Art. 16"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p93",
    "variant": "clean",
    "text": "Processing is opaque and misleading to users.",
    "expected": [
      "GDPR Art. 5(1)(a)"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p93",
    "variant": "adv",
    "text": "The experience here isn't fully transparent.",
    "expected": [
      "GDPR Art. 5(1)(a)"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p94",
    "variant": "clean",
    "text": "Retention periods are hidden from data subjects.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p94",
    "variant": "adv",
    "text": "We keep things a bit vague on timelines.",
    "expected": [
      "GDPR Art. 13/14"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p95",
    "variant": "clean",
    "text": "Sub-processors operate without any contract.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p95",
    "variant": "adv",
    "text": "Some downstream work happens informally.",
    "expected": [
      "GDPR Art. 28"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p96",
    "variant": "clean",
    "text": "Export functionality was quietly removed.",
    "expected": [
      "GDPR Art. 20"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p96",
    "variant": "adv",
    "text": "That feature isn't really available right now.",
    "expected": [
      "GDPR Art. 20"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p97",
    "variant": "clean",
    "text": "Mitigations identified in the DPIA were never applied.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p97",
    "variant": "adv",
    "text": "Some of the recommended follow-ups are still open.",
    "expected": [
      "GDPR Art. 35"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p98",
    "variant": "clean",
    "text": "Logging of access to personal data was disabled.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p98",
    "variant": "adv",
    "text": "Monitoring on that system is currently limited.",
    "expected": [
      "GDPR Art. 32"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p99",
    "variant": "clean",
    "text": "Data subject requests are logged and tracked to completion.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p99",
    "variant": "adv",
    "text": "Requests from users go into a queue that isn't closely monitored.",
    "expected": [
      "GDPR Art. 15"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p100",
    "variant": "clean",
    "text": "Processing continues without a documented lawful basis in the RoPA.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p100",
    "variant": "adv",
    "text": "No specific reason is on file.",
    "expected": [
      "GDPR Art. 6"
    ],
    "attack_type": "keyword_removal"
  },
  {
    "pair_id": "p101",
    "variant": "clean",
    "text": "A personal data breach occurred, the authority was not notified, and affected users were not informed.",
    "expected": [
      "GDPR Art. 33",
      "GDPR Art. 34"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p101",
    "variant": "adv",
    "text": "A security incident happened, and nobody outside the team knew about it.",
    "expected": [
      "GDPR Art. 33",
      "GDPR Art. 34"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p102",
    "variant": "clean",
    "text": "We transfer personal data to a US processor without SCCs or a signed data processing agreement.",
    "expected": [
      "GDPR Ch. V",
      "GDPR Art. 28"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p102",
    "variant": "adv",
    "text": "Data moves to an overseas partner without the usual paperwork in place.",
    "expected": [
      "GDPR Ch. V",
      "GDPR Art. 28"
    ],
    "attack_type": "paraphrase"
  },
  {
    "pair_id": "p103",
    "variant": "clean",
    "text": "High-risk profiling is performed without a DPIA, and the activity is not logged in the processing register.",
    "expected": [
      "GDPR Art. 35",
      "GDPR Art. 30"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p103",
    "variant": "adv",
    "text": "The new scoring feature went live without the usual risk paperwork or tracking.",
    "expected": [
      "GDPR Art. 35",
      "GDPR Art. 30"
    ],
    "attack_type": "obfuscation"
  },
  {
    "pair_id": "p104",
    "variant": "clean",
    "text": "Users cannot withdraw consent, and their erasure requests are also ignored.",
    "expected": [
      "GDPR Art. 7",
      "GDPR Art. 17"
    ],
    "attack_type": "none"
  },
  {
    "pair_id": "p104",
    "variant": "adv",
    "text": "Permission removal and deletion asks both go nowhere.",
    "expected": [
      "GDPR Art. 7",
      "GDPR Art. 17"
    ],
    "attack_type": "keyword_removal"
  }
]

In [9]:
df_eval = pd.DataFrame(eval_data)
print(f"Evaluation dataset: {len(df_eval)} cases, {df_eval['pair_id'].nunique()} pairs")
print(df_eval['attack_type'].value_counts())

Evaluation dataset: 208 cases, 104 pairs
attack_type
none               104
obfuscation         27
paraphrase          26
keyword_removal     26
negation            25
Name: count, dtype: int64


## 6. Evaluation Metrics

Precision/Recall@K plus rank-aware MRR and nDCG@5, computed against the full similarity ranking so gating doesn't artificially truncate the rank-aware metrics.

In [10]:
def precision_recall(expected, predicted):
    expected, predicted = set(expected), set(predicted)
    tp = len(expected & predicted)
    fp = len(predicted - expected)
    fn = len(expected - predicted)
    precision = tp / len(predicted) if predicted else 0.0
    recall = tp / len(expected) if expected else 0.0
    return precision, recall, fp, fn

def reciprocal_rank(expected, ranked_list):
    expected = set(expected)
    for i, cid in enumerate(ranked_list, start=1):
        if cid in expected:
            return 1.0 / i
    return 0.0

def ndcg_at_k(expected, ranked_list, k=5):
    expected = set(expected)
    dcg = sum((1.0 if cid in expected else 0.0) / np.log2(i + 1) for i, cid in enumerate(ranked_list[:k], start=1))
    ideal_hits = min(len(expected), k)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_engine(engine, df_eval, top_k=5, rank_k=5):
    rows = []
    for _, row in df_eval.iterrows():
        result = engine.assess(row["text"], top_k=top_k)
        expected = row["expected"]
        predicted = result["controls"]
        precision, recall, fp, fn = precision_recall(expected, predicted)
        mrr = reciprocal_rank(expected, result["full_ranked_controls"])
        ndcg = ndcg_at_k(expected, result["full_ranked_controls"], k=rank_k)
        rows.append({
            "pair_id": row["pair_id"], "variant": row["variant"], "attack_type": row["attack_type"],
            "domain": result["domain"], "confidence": result["confidence"],
            "controls": predicted, "expected": list(expected),
            "precision": precision, "recall": recall, "fp_count": fp, "missed_count": fn,
            "mrr": mrr, "ndcg": ndcg,
        })
    return pd.DataFrame(rows)

def clean_performance(df_results):
    clean = df_results[df_results["variant"] == "clean"]
    return {
        "precision": clean["precision"].mean(), "recall": clean["recall"].mean(),
        "mrr": clean["mrr"].mean(), "ndcg@5": clean["ndcg"].mean(),
        "avg_fp": clean["fp_count"].mean(),
    }

def pairwise_attack_metrics(df_results):
    wide = df_results.pivot(index="pair_id", columns="variant",
                             values=["domain", "controls", "recall", "fp_count", "attack_type"])
    wide.columns = ["_".join(c).strip() for c in wide.columns]
    wide["attack_type"] = wide["attack_type_adv"]

    wide["domain_drift"] = wide["domain_clean"] != wide["domain_adv"]
    wide["recall_drop"] = (wide["recall_clean"] == 1.0) & (wide["recall_adv"] < 1.0)
    wide["fp_increase"] = wide["fp_count_adv"] > wide["fp_count_clean"]
    wide["attack_success"] = wide["domain_drift"] | wide["recall_drop"] | wide["fp_increase"]

    def jaccard(a, b):
        a, b = set(a), set(b)
        return 1.0 if not a and not b else len(a & b) / len(a | b)

    wide["control_overlap"] = wide.apply(lambda r: jaccard(r["controls_clean"], r["controls_adv"]), axis=1)
    return wide

def summarize_attack_metrics(wide):
    return {
        "attack_success_rate": wide["attack_success"].mean(),
        "domain_drift_rate": wide["domain_drift"].mean(),
        "recall_drop_rate": wide["recall_drop"].mean(),
        "fp_increase_rate": wide["fp_increase"].mean(),
        "mean_control_overlap": wide["control_overlap"].mean(),
    }

print("Metric functions defined")

Metric functions defined


## 7. Run Evaluation for All Four Conditions

In [11]:
RESULTS = {}       # name -> per-case df from evaluate_engine
WIDE = {}           # name -> paired clean/adv df from pairwise_attack_metrics
SUMMARY = {}        # name -> dict of attack metrics
CLEAN = {}          # name -> dict of clean-input metrics

for name, engine in ENGINES.items():
    df_res = evaluate_engine(engine, df_eval)
    RESULTS[name] = df_res
    WIDE[name] = pairwise_attack_metrics(df_res)
    SUMMARY[name] = summarize_attack_metrics(WIDE[name])
    CLEAN[name] = clean_performance(df_res)

print("Evaluation complete for:", list(RESULTS.keys()))

Evaluation complete for: ['baseline', 'retrieval_only', 'defended']


## 8. Table 6 (revised) — Overall Attack Impact: Baseline vs. Defended

In [12]:
table6 = pd.DataFrame({
    "Baseline": SUMMARY["baseline"],
    "Defended": SUMMARY["defended"],
}).round(3)
table6

,Baseline,Defended
attack_success_rate,0.500,0.433
domain_drift_rate,0.500,0.433
recall_drop_rate,0.260,0.212
fp_increase_rate,0.298,0.212
mean_control_overlap,0.510,0.577


## 9. Table 7 (revised) — Impact by Attack Type (N per type now >=25, not n=1/2)

In [13]:
def by_attack_type(wide):
    g = wide.groupby("attack_type").agg(
        n=("attack_success", "size"),
        drift=("domain_drift", "mean"),
        recall_drop=("recall_drop", "mean"),
        fp_increase=("fp_increase", "mean"),
        overlap=("control_overlap", "mean"),
    ).round(3)
    return g

table7_baseline = by_attack_type(WIDE["baseline"])
table7_defended = by_attack_type(WIDE["defended"])
print("Baseline by attack type:")
display(table7_baseline)
print("\nDefended by attack type:")
display(table7_defended)

Baseline by attack type:


,n,drift,recall_drop,fp_increase,overlap
attack_type,,,,,
keyword_removal,26,0.692,0.385,0.308,0.327
negation,25,0.280,0.080,0.160,0.720
obfuscation,27,0.593,0.333,0.407,0.407
paraphrase,26,0.423,0.231,0.308,0.596



Defended by attack type:


,n,drift,recall_drop,fp_increase,overlap
attack_type,,,,,
keyword_removal,26,0.500,0.385,0.154,0.500
negation,25,0.240,0.040,0.160,0.760
obfuscation,27,0.667,0.333,0.333,0.352
paraphrase,26,0.308,0.077,0.192,0.712


## 10. Clean-Input Performance — All Evaluated Conditions

In [14]:
table_clean = pd.DataFrame(CLEAN).round(3).T
table_clean

,precision,recall,mrr,ndcg@5,avg_fp
baseline,0.517,0.678,0.851,0.865,0.904
retrieval_only,0.205,0.904,0.852,0.879,3.702
defended,0.507,0.716,0.852,0.879,1.077


## 11. Three-Way Defense Ablation

Isolates whether the improvement comes from training-time semantic hardening, retrieval-time domain gating, their combination, or neither.

In [15]:
table8 = pd.DataFrame(SUMMARY).round(3).T
table8 = table8[["attack_success_rate", "domain_drift_rate", "recall_drop_rate", "fp_increase_rate", "mean_control_overlap"]]
table8

,attack_success_rate,domain_drift_rate,recall_drop_rate,fp_increase_rate,mean_control_overlap
baseline,0.500,0.500,0.260,0.298,0.510
retrieval_only,0.635,0.500,0.240,0.317,0.400
defended,0.433,0.433,0.212,0.212,0.577


## 12. Rank-Aware Metrics (MRR / nDCG@5)

In [16]:
rank_metrics = {}
for name, df_res in RESULTS.items():
    clean = df_res[df_res["variant"] == "clean"]
    adv = df_res[df_res["variant"] == "adv"]
    rank_metrics[name] = {
        "MRR (clean)": clean["mrr"].mean(), "MRR (adv)": adv["mrr"].mean(),
        "nDCG@5 (clean)": clean["ndcg"].mean(), "nDCG@5 (adv)": adv["ndcg"].mean(),
    }
pd.DataFrame(rank_metrics).round(3).T

,MRR (clean),MRR (adv),nDCG@5 (clean),nDCG@5 (adv)
baseline,0.851,0.678,0.865,0.715
retrieval_only,0.852,0.712,0.879,0.748
defended,0.852,0.712,0.879,0.748


## 13. McNemar's Exact Test — Baseline vs. Defended Attack Success (paired by case)\n\nTests whether the drop in attack success rate (baseline vs. defended) is statistically significant, not just numerically different.

In [17]:
from scipy.stats import wilcoxon

aligned_metrics = WIDE["baseline"][["control_overlap","domain_drift","recall_drop","fp_increase"]].join(
    WIDE["defended"][["control_overlap","domain_drift","recall_drop","fp_increase"]],
    lsuffix="_base", rsuffix="_def"
)

# --- Test 1: Wilcoxon signed-rank on control overlap (continuous, paired) ---
stat, p_wilcoxon = wilcoxon(aligned_metrics["control_overlap_def"], aligned_metrics["control_overlap_base"])
print(f"Wilcoxon signed-rank (control overlap, defended vs baseline): stat={stat:.1f}, p={p_wilcoxon:.4f}")
print("  => significant at alpha=0.05" if p_wilcoxon < 0.05 else "  => not significant at alpha=0.05")

# --- Test 2: Paired bootstrap 95% CI on attack_success_rate difference ---
np.random.seed(42)
n = len(WIDE["baseline"])
succ_base = WIDE["baseline"]["attack_success"].values.astype(int)
succ_def = WIDE["defended"]["attack_success"].values.astype(int)
boot_diffs = []
for _ in range(5000):
    idx = np.random.choice(n, n, replace=True)
    boot_diffs.append(succ_base[idx].mean() - succ_def[idx].mean())
boot_diffs = np.array(boot_diffs)
ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
observed = succ_base.mean() - succ_def.mean()
print(f"\nBootstrap 95% CI, attack_success_rate (baseline - defended): [{ci_low:.3f}, {ci_high:.3f}]  (observed = {observed:.3f})")
print("  => CI excludes 0, consistent with a real effect" if ci_low > 0 or ci_high < 0 else "  => CI includes 0, effect not distinguishable from noise at this sample size")

# --- Test 3: Omnibus composite permutation test (combines all 3 attack components per pair, not just one) ---
composite_base = (aligned_metrics["domain_drift_base"].astype(int) + aligned_metrics["recall_drop_base"].astype(int)
                   + aligned_metrics["fp_increase_base"].astype(int))
composite_def = (aligned_metrics["domain_drift_def"].astype(int) + aligned_metrics["recall_drop_def"].astype(int)
                  + aligned_metrics["fp_increase_def"].astype(int))
observed_diff = (composite_base - composite_def).mean()

rng = np.random.default_rng(42)
combined = np.vstack([composite_base.values, composite_def.values]).T
perm_diffs = []
for _ in range(10000):
    flips = rng.integers(0, 2, size=n).astype(bool)
    a = np.where(flips, combined[:, 1], combined[:, 0])
    b = np.where(flips, combined[:, 0], combined[:, 1])
    perm_diffs.append((a - b).mean())
perm_diffs = np.array(perm_diffs)
p_perm = (np.abs(perm_diffs) >= abs(observed_diff)).mean()

print(f"\nOmnibus composite permutation test (drift+recall_drop+fp_increase combined per pair):")
print(f"  observed mean improvement = {observed_diff:.3f} fewer failure-components per pair, p = {p_perm:.4f}")
print("  => significant at alpha=0.05" if p_perm < 0.05 else "  => not significant at alpha=0.05")

Wilcoxon signed-rank (control overlap, defended vs baseline): stat=110.0, p=0.1276
  => not significant at alpha=0.05

Bootstrap 95% CI, attack_success_rate (baseline - defended): [-0.029, 0.154]  (observed = 0.067)
  => CI includes 0, effect not distinguishable from noise at this sample size

Omnibus composite permutation test (drift+recall_drop+fp_increase combined per pair):
  observed mean improvement = 0.202 fewer failure-components per pair, p = 0.0820
  => not significant at alpha=0.05


In [18]:
def mcnemar_test(success_a, success_b):
    """success_a/b: aligned boolean Series, e.g. baseline vs defended attack_success per pair.
    Returns (b, c, p) where b = a-success/b-fail, c = a-fail/b-success."""
    b = int(((success_a == True) & (success_b == False)).sum())
    c = int(((success_a == False) & (success_b == True)).sum())
    n = b + c
    p = 1.0 if n == 0 else binomtest(min(b, c), n, 0.5, alternative="two-sided").pvalue
    return b, c, p

aligned = WIDE["baseline"][["attack_success"]].join(
    WIDE["defended"][["attack_success"]], lsuffix="_baseline", rsuffix="_defended"
)
b, c, p = mcnemar_test(aligned["attack_success_baseline"], aligned["attack_success_defended"])
print(f"Discordant pairs: baseline-fails/defended-succeeds={b}, baseline-succeeds/defended-fails={c}")
print(f"McNemar's exact test p-value = {p:.4f}")
print("=> statistically significant at alpha=0.05" if p < 0.05 else "=> NOT statistically significant at alpha=0.05")

Discordant pairs: baseline-fails/defended-succeeds=15, baseline-succeeds/defended-fails=8
McNemar's exact test p-value = 0.2100
=> NOT statistically significant at alpha=0.05


## 14. Keyword-Weighted Retrieval — Lightweight Field-Weighting Comparison

Not a full BM25F implementation — this up-weights the `keywords` field in the control text used for embedding, as a lower-effort proxy for field-weighted scoring, evaluated as an additional ablation on top of the defended configuration.

In [19]:
defended_kw_weighted_engine = SBERTGRCEngine(
    gdpr_controls, train_df, encoder,
    use_augmented_training=True, gate_top_k=3, control_text_fn=control_text_keyword_weighted, clf_C=0.5,
)

df_res_kw = evaluate_engine(defended_kw_weighted_engine, df_eval)
wide_kw = pairwise_attack_metrics(df_res_kw)
summary_kw = summarize_attack_metrics(wide_kw)
clean_kw = clean_performance(df_res_kw)

comparison = pd.DataFrame({
    "Defended (enriched control text)": {**SUMMARY["defended"], **{f"clean_{k}": v for k, v in CLEAN["defended"].items()}},
    "Defended (keyword-weighted control text)": {**summary_kw, **{f"clean_{k}": v for k, v in clean_kw.items()}},
}).round(3)
comparison

,Defended (enriched control text),Defended (keyword-weighted control text)
attack_success_rate,0.433,0.606
domain_drift_rate,0.433,0.433
recall_drop_rate,0.212,0.192
fp_increase_rate,0.212,0.337
mean_control_overlap,0.577,0.407
clean_precision,0.507,0.217
clean_recall,0.716,0.904
clean_mrr,0.852,0.838
clean_ndcg@5,0.879,0.861
clean_avg_fp,1.077,3.481


## 15. Audit Logging (unchanged from prior version, retained for traceability)

In [20]:
import time, uuid, hashlib
from datetime import datetime, timezone
from pathlib import Path

AUDIT_PATH = Path("audit_log.jsonl")

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def append_audit_event(event: dict, path: Path = AUDIT_PATH) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False) + "\n")

def assess_with_audit(engine, risk_text: str, actor: str = "anonymous", top_k: int = 5):
    request_id = str(uuid.uuid4())
    start = time.time()
    event = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "request_id": request_id,
        "actor": actor, "engine": type(engine).__name__, "engine_version": "2.0",
        "top_k": top_k, "risk_text_hash": sha256_text(risk_text), "status": "started",
    }
    append_audit_event(event)
    try:
        result = engine.assess(risk_text, top_k=top_k)
        latency_ms = int((time.time() - start) * 1000)
        append_audit_event({
            "timestamp_utc": datetime.now(timezone.utc).isoformat(), "request_id": request_id,
            "status": "success", "latency_ms": latency_ms,
            "predicted_domain": result.get("domain"), "confidence": result.get("confidence"),
            "controls": result.get("controls"), "top_domains": result.get("top_domains"),
        })
        return result
    except Exception as e:
        append_audit_event({
            "timestamp_utc": datetime.now(timezone.utc).isoformat(), "request_id": request_id,
            "status": "failure", "latency_ms": int((time.time() - start) * 1000), "error": repr(e),
        })
        raise

print(assess_with_audit(defended_engine, "No lawful basis for processing personal data", actor="auditor_01"))

{'domain': 'LawfulBasis', 'confidence': 0.12391041556691251, 'top_domains': ['LawfulBasis', 'Rights', 'Principles'], 'controls': ['GDPR Art. 6', 'GDPR Art. 7'], 'ranked_controls': ['GDPR Art. 6', 'GDPR Art. 7'], 'full_ranked_controls': ['GDPR Art. 6', 'GDPR Art. 5(1)(a)', 'GDPR Art. 7', 'GDPR Art. 32', 'GDPR Art. 16', 'GDPR Art. 13/14', 'GDPR Art. 20', 'GDPR Art. 15', 'GDPR Art. 30', 'GDPR Art. 35', 'GDPR Art. 28', 'GDPR Ch. V', 'GDPR Art. 33', 'GDPR Art. 24', 'GDPR Art. 17', 'GDPR Art. 34']}


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfEncoder:
    def __init__(self, corpus):
        self.vec = TfidfVectorizer(ngram_range=(1,2), max_features=200)
        self.vec.fit(corpus)
    def encode(self, texts):
        return self.vec.transform(texts).toarray()

corpus = train_df["text"].tolist() + [control_text_simple(c) for c in gdpr_controls]
tfidf_encoder = TfidfEncoder(corpus)

tfidf_baseline_engine = SBERTGRCEngine(
    gdpr_controls, train_df, tfidf_encoder,
    use_augmented_training=False, gate_top_k=1, control_text_fn=control_text_simple,
)
tfidf_defended_engine = SBERTGRCEngine(
    gdpr_controls, train_df, tfidf_encoder,
    use_augmented_training=True, gate_top_k=1, control_text_fn=control_text_enriched, clf_C=0.5,
)

df_res_tb = evaluate_engine(tfidf_baseline_engine, df_eval)
df_res_td = evaluate_engine(tfidf_defended_engine, df_eval)
wide_tb = pairwise_attack_metrics(df_res_tb)
wide_td = pairwise_attack_metrics(df_res_td)
summ_tb = summarize_attack_metrics(wide_tb)
summ_td = summarize_attack_metrics(wide_td)
clean_tb = clean_performance(df_res_tb)
clean_td = clean_performance(df_res_td)

tfidf_vs_sbert = pd.DataFrame({
    "TF-IDF baseline": {**summ_tb, **{f"clean_{k}": v for k, v in clean_tb.items()}},
    "TF-IDF defended": {**summ_td, **{f"clean_{k}": v for k, v in clean_td.items()}},
    "SBERT baseline": {**SUMMARY["baseline"], **{f"clean_{k}": v for k, v in CLEAN["baseline"].items()}},
    "SBERT defended": {**SUMMARY["defended"], **{f"clean_{k}": v for k, v in CLEAN["defended"].items()}},
}).round(3)
tfidf_vs_sbert

,TF-IDF baseline,TF-IDF defended,SBERT baseline,SBERT defended
attack_success_rate,0.625,0.490,0.500,0.433
domain_drift_rate,0.625,0.490,0.500,0.433
recall_drop_rate,0.308,0.250,0.260,0.212
fp_increase_rate,0.308,0.163,0.298,0.212
mean_control_overlap,0.380,0.514,0.510,0.577
clean_precision,0.394,0.442,0.517,0.507
clean_recall,0.582,0.678,0.678,0.716
clean_mrr,0.796,0.795,0.851,0.852
clean_ndcg@5,0.799,0.801,0.865,0.879
clean_avg_fp,1.298,1.279,0.904,1.077


In [22]:
topk_rows = []
for name, eng in ENGINES.items():
    for k in [1, 3, 5]:
        df_res_k = evaluate_engine(eng, df_eval, top_k=k)
        cp = clean_performance(df_res_k)
        topk_rows.append({"engine": name, "top_k": k, **cp})

df_topk = pd.DataFrame(topk_rows).round(3)
df_topk

,engine,top_k,precision,recall,mrr,ndcg@5,avg_fp
0,baseline,1,0.673,0.654,0.851,0.865,0.327
1,baseline,3,0.522,0.678,0.851,0.865,0.837
2,baseline,5,0.517,0.678,0.851,0.865,0.904
3,retrieval_only,1,0.769,0.750,0.852,0.879,0.231
4,retrieval_only,3,0.311,0.894,0.852,0.879,2.067
5,retrieval_only,5,0.205,0.904,0.852,0.879,3.702
6,defended,1,0.702,0.683,0.852,0.879,0.298
7,defended,3,0.518,0.716,0.852,0.879,0.923
8,defended,5,0.507,0.716,0.852,0.879,1.077


In [23]:
class TfidfEncoder:
    def __init__(self, corpus):
        self.vec = TfidfVectorizer(ngram_range=(1,2), max_features=200)
        self.vec.fit(corpus)
    def encode(self, texts):
        return self.vec.transform(texts).toarray()

corpus = train_df["text"].tolist() + [control_text_simple(c) for c in gdpr_controls]
tfidf_encoder = TfidfEncoder(corpus)

tfidf_baseline_engine = SBERTGRCEngine(
    gdpr_controls, train_df, tfidf_encoder,
    use_augmented_training=False, gate_top_k=1, control_text_fn=control_text_simple,
)
tfidf_defended_engine = SBERTGRCEngine(
    gdpr_controls, train_df, tfidf_encoder,
    use_augmented_training=True, gate_top_k=1, control_text_fn=control_text_enriched, clf_C=0.5,
)

df_res_tb = evaluate_engine(tfidf_baseline_engine, df_eval)
df_res_td = evaluate_engine(tfidf_defended_engine, df_eval)
wide_tb = pairwise_attack_metrics(df_res_tb)
wide_td = pairwise_attack_metrics(df_res_td)
summ_tb = summarize_attack_metrics(wide_tb)
summ_td = summarize_attack_metrics(wide_td)
clean_tb = clean_performance(df_res_tb)
clean_td = clean_performance(df_res_td)

tfidf_vs_sbert = pd.DataFrame({
    "TF-IDF baseline": {**summ_tb, **{f"clean_{k}": v for k, v in clean_tb.items()}},
    "TF-IDF defended": {**summ_td, **{f"clean_{k}": v for k, v in clean_td.items()}},
    "SBERT baseline": {**SUMMARY["baseline"], **{f"clean_{k}": v for k, v in CLEAN["baseline"].items()}},
    "SBERT defended": {**SUMMARY["defended"], **{f"clean_{k}": v for k, v in CLEAN["defended"].items()}},
}).round(3)
tfidf_vs_sbert

,TF-IDF baseline,TF-IDF defended,SBERT baseline,SBERT defended
attack_success_rate,0.625,0.490,0.500,0.433
domain_drift_rate,0.625,0.490,0.500,0.433
recall_drop_rate,0.308,0.250,0.260,0.212
fp_increase_rate,0.308,0.163,0.298,0.212
mean_control_overlap,0.380,0.514,0.510,0.577
clean_precision,0.394,0.442,0.517,0.507
clean_recall,0.582,0.678,0.678,0.716
clean_mrr,0.796,0.795,0.851,0.852
clean_ndcg@5,0.799,0.801,0.865,0.879
clean_avg_fp,1.298,1.279,0.904,1.077


In [24]:
joined_drift = WIDE["baseline"][["attack_type", "domain_drift"]].join(
    WIDE["defended"][["domain_drift"]], lsuffix="_base", rsuffix="_def"
)

def bootstrap_ci_diff(a, b, n_boot=5000, seed=42):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    n = len(a)
    rng = np.random.default_rng(seed)
    diffs = np.array([a[rng.integers(0, n, n)].mean() - b[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    return a.mean() - b.mean(), np.percentile(diffs, [2.5, 97.5])

rows = []
for atype, grp in joined_drift.groupby("attack_type"):
    obs, (lo, hi) = bootstrap_ci_diff(grp["domain_drift_base"], grp["domain_drift_def"])
    rows.append({
        "attack_type": atype, "n": len(grp),
        "baseline_drift": grp["domain_drift_base"].mean(), "defended_drift": grp["domain_drift_def"].mean(),
        "diff": obs, "ci_95_low": lo, "ci_95_high": hi,
    })

table_subgroup_ci = pd.DataFrame(rows).round(3)
table_subgroup_ci

,attack_type,n,baseline_drift,defended_drift,diff,ci_95_low,ci_95_high
0,keyword_removal,26,0.692,0.500,0.192,-0.077,0.462
1,negation,25,0.280,0.240,0.040,-0.200,0.280
2,obfuscation,27,0.593,0.667,-0.074,-0.333,0.185
3,paraphrase,26,0.423,0.308,0.115,-0.154,0.385
